# EyeCan Logo Detector

End-to-end pipeline for the [eyecan.ai](https://eyecan.ai) logo detection challenge.

The repository is structured as follows:

```
eyecan-logo-detector/
├── src/
│   ├── dataset_generator.py   # Module B: synthetic data generation
│   ├── logo_detector.py       # Module A: training & inference
│   └── test_inference.py      # batch testing with visualization
├── logos/                     # PNG logo variants (RGBA or RGB)
├── sample_background/         # ~1000 bundled background images
├── checkpoints/
│   └── best_20k.pt            # pre-trained checkpoint (20k samples)
├── test_images/               # place your real-world photos here
└── test_results/              # annotated predictions (auto-created)
```

**Workflow:**
1. **Environment Setup**: clone the repo and install dependencies.
2. **Dataset Generation**: composite logos onto backgrounds to build a synthetic dataset.
3. **Training**: fine-tune MobileNetV3-Small on the generated dataset.
4. **Inference**: run the detector on your own images and visualize results.

## 1. Environment Setup

In [ ]:
# Clone the repository and move into it
!git clone https://github.com/itsjustwhitee/eyecan-logo-detector.git
%cd eyecan-logo-detector

In [ ]:
# Install dependencies
# Note: install the PyTorch version matching your hardware (NVIDIA/AMD/Intel)
# before running this cell if you want GPU acceleration.
!pip install pipelime-python==2.2.0 albumentations opencv-python numpy tqdm

## 2. Dataset Generation

Generates synthetic training images by compositing the Eyecan logo onto random backgrounds.
The `sample_background/` folder (included in the repo, ~1000 images) is used here for a quick run.
For a full-scale dataset, replace `--bg_dir` with `backgrounds/Images` (full [MIT Indoor Scenes](https://www.kaggle.com/datasets/itsahmad/indoor-scenes-cvpr-2019) dataset, not included).

Each generated sample consists of a composited JPEG image and a JSON file with the normalised logo centroid coordinates `{"x": ..., "y": ...}` ∈ [0, 1].

In [ ]:
!python src/dataset_generator.py \
    --bg_dir      sample_background \
    --logos_dir   logos \
    --num_samples 1000 \
    --output      generated_dataset \
    --batch_size  200 \
    --num_workers 0 \
    --seed        42

## 3. Training

Trains a MobileNetV3-Small backbone with a custom regression head to predict the normalised (x, y) centroid of the logo.

> **Note:** `--epochs 10` is intentionally low for a quick demo run in this notebook. For a properly trained model use `--epochs 100` (early stopping will kick in around epoch 30–45). The pre-trained `best_20k.pt` checkpoint is available if you want to skip training entirely (go directly to Step 4).

In [ ]:
!python src/logo_detector.py --train \
    --dataset_dir generated_dataset \
    --output_dir  checkpoints \
    --epochs      10 \
    --lr          1e-3 \
    --batch_size  32 \
    --seed        0

## 4. Inference

Place your test images in the `test_images/` folder, then run the cell below.

You can choose which checkpoint to use:

- **`checkpoints/best_20k.pt`** — pre-trained on 20k synthetic images, included in the repo. Use this for a robust out-of-the-box evaluation.
- **`checkpoints/best.pt`** — the best checkpoint produced by your own training run above.

The script saves annotated images (red crosshair on the predicted centroid) to `test_results/`, then the Matplotlib block below displays them inline.

In [ ]:
# Run batch inference — saves annotated images to test_results/
!python src/test_inference.py \
    --checkpoint checkpoints/best_20k.pt \
    --input      test_images \
    --output     test_results

In [ ]:
# Display results inline (works in headless environments: Colab, JupyterHub, etc.)
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path("test_results")
result_images = sorted(
    list(results_dir.glob("*.jpg"))
    + list(results_dir.glob("*.jpeg"))
    + list(results_dir.glob("*.png"))
)

if not result_images:
    print("No results found in test_results/. Add photos to test_images/ and re-run.")
else:
    print(f"Visualizing {len(result_images)} result(s)...")
    fig, axes = plt.subplots(len(result_images), 1,
                            figsize=(10, 8 * len(result_images)))
    if len(result_images) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, result_images):
        bgr = cv2.imread(str(img_path))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        ax.imshow(rgb)
        ax.axis("off")
        ax.set_title(img_path.name, fontsize=14)

    plt.tight_layout()
    plt.show()